In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command


class State(TypedDict):
    message: str
    decision: str


def create_request(state: State):
    print("Request created.")

    return {
        "message": "Do you want to publish this post?"
    }


def human_approval(state: State):

    decision = interrupt({
        "message": state["message"],
        "options": ["accept", "reject"]
    })

    return {
        "decision": decision
    }


def final_result(state: State):

    if state["decision"] == "accept":
        print("Accepted! Post published.")

    elif state["decision"] == "reject":
        print("Rejected! Post was not published.")

    return state


# Create graph
builder = StateGraph(State)

builder.add_node("create_request", create_request)
builder.add_node("human_approval", human_approval)
builder.add_node("final_result", final_result)

builder.add_edge(START, "create_request")
builder.add_edge("create_request", "human_approval")
builder.add_edge("human_approval", "final_result")
builder.add_edge("final_result", END)

graph = builder.compile()

In [14]:
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)

In [15]:
config = {
    "configurable": {
        "thread_id": "1"
    }
}

result = graph.invoke(
    {
        "message": "",
        "decision": ""
    },
    config
)

print(result)

Request created.
{'message': 'Do you want to publish this post?', 'decision': '', '__interrupt__': [Interrupt(value={'message': 'Do you want to publish this post?', 'options': ['accept', 'reject']}, id='549d7f78db2c16b5084b9433a493347b')]}


In [16]:
graph.invoke(
    Command(resume="accept"),
    config
)

Accepted! Post published.


{'message': 'Do you want to publish this post?', 'decision': 'accept'}

In [17]:
graph.invoke(
    Command(resume="reject"),
    config
)

{'message': 'Do you want to publish this post?', 'decision': 'accept'}

In [18]:
graph.invoke(
    Command(resume=""),
    config
)

{'message': 'Do you want to publish this post?', 'decision': 'accept'}

In [19]:
graph.invoke(
    Command(resume="reject"),
    config
)

{'message': 'Do you want to publish this post?', 'decision': 'accept'}